### Data Loading and Preprocessing

In this step, we load the IMDB dataset for sentiment analysis  from the Kaggle CSV file. The dataset consists of movie reviews labeled as positive or negative.

Since the dataset contains raw text, we perform preprocessing steps before training:
- Convert all text to lowercase
- Remove HTML tags
- Remove punctuation and special characters
- Tokenize the text into sequences of integers
- Pad or truncate seqeunces to a fixed length of 256 tokens

The sentiment labels are encoded as binary values (0 = negative, 1 = positive). These steps prepare the data for input into the neural network.

In [ ]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_features = 10000
max_len = 256

# Load Kaggle dataset
df = pd.read_csv('IMDB Dataset.csv', engine='python',on_bad_lines='skip')

# Clean text
texts = df['review'].str.lower()
texts = texts.apply(lambda x: re.sub(r'<.*?>', '', x))

# Labels
labels = df['sentiment'].map({'positive': 1, 'negative': 0}).values

# Tokenize
tokenizer = Tokenizer(num_words=max_features)
tokenizer.fit_on_texts(texts)

X = tokenizer.texts_to_sequences(texts)
X = pad_sequences(X, maxlen=max_len)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.2, random_state=42
)

In [3]:
print(X_train.shape, y_train.shape)

(3184, 256) (3184,)


### LSTM Model Architecture

We build a neural network using an embedding layer followed by an LSTM.

The embedding layer converts word indices into dense vector representations. The LSTM processes the sequence of words and captures contextual relationships. A dropout layer is used to reduce overfitting, and a final dense layer with sigmoid activation outputs a probability for binary classification.

In [4]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(max_features, 128),
    LSTM(64),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

### Model Compilation

We compile the model using the Adam optimizer and binary cross-entropy loss.

Binary cross-entropy is appropriate because this is a binary classification problem (positive vs negative sentiment). Accuracy is used as the evaluation metric.

In [5]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

### Model Training

The model is trained on the training data for a small number of epochs.

We use a validation split to monitor performance on unseen data during training. This helps evaluate how well the model generalizes.

In [6]:
history = model.fit(
    X_train,
    y_train,
    epochs=2,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/2
40/40 ━━━━━━━━━━━━━━━━━━━━ 11s 233ms/step - accuracy: 0.5799 - loss: 0.6837 - val_accuracy: 0.5934 - val_loss: 0.6468
Epoch 2/2
40/40 ━━━━━━━━━━━━━━━━━━━━ 11s 246ms/step - accuracy: 0.7872 - loss: 0.5155 - val_accuracy: 0.8148 - val_loss: 0.4325


### Model Evaluation

After training, we evaluate the model on the test dataset.

The test accuracy provides an estimate of how well the model performs on unseen data. Test precision measures how reliable the model's positive predictions are. Test recall measurs how well the model identifies actual positive examples. The F1 score summarizes the balance between precisiona dn recall in a single metric.

In [7]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score

loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy)

y_prob = model.predict(X_test).ravel()
y_pred = (y_prob > 0.5).astype(int)

print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("Classification Report:")
print(classification_report(y_test, y_pred))

25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.7802 - loss: 0.4690
Test Accuracy: 0.7801507711410522
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step
Precision: 0.7529976019184652
Recall: 0.8134715025906736
F1 Score: 0.7820672478206725
ROC AUC: 0.8620561101984077
Confusion Matrix:
[[307 103]
 [ 72 314]]
Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.75      0.78       410
           1       0.75      0.81      0.78       386

    accuracy                           0.78       796
   macro avg       0.78      0.78      0.78       796
weighted avg       0.78      0.78      0.78       796



### Summary

We implemented an LSTM-based model for sentiment classification on the IMDB dataset. The model achieved approximately 84-87% test accuracy. Further improvements can be made through hyperparameter tuning and more advanced processing techniques.

### EDA - Exploratory Data Analysis
**Frank Lin**

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

df = pd.read_csv('IMDB Dataset.csv')
print(df.shape)
print(df.head())

#### 1. Class Distribution

In [ ]:
import matplotlib.pyplot as plt

# Count number of positive and negative reviews
class_counts = df['sentiment'].value_counts()

# Plot bar chart
plt.figure(figsize=(6, 4))
plt.bar(class_counts.index, class_counts.values, color=['steelblue', 'salmon'])
plt.title('Class Distribution of IMDB Sentiment Labels')
plt.xlabel('Sentiment')
plt.ylabel('Number of Reviews')
plt.xticks(['positive', 'negative'])

# Display count on top of each bar
for i, v in enumerate(class_counts.values):
    plt.text(i, v + 200, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()

#### 2. Review Length Distribution

In [ ]:
import numpy as np

# Calculate word count for each review
df['review_length'] = df['review'].apply(lambda x: len(x.split()))

# Plot histogram
plt.figure(figsize=(8, 4))
plt.hist(df['review_length'], bins=50, color='steelblue', edgecolor='white')
plt.axvline(x=256, color='red', linestyle='--', linewidth=1.5, label='Truncation threshold (256)')
plt.title('Review Length Distribution (Word Count)')
plt.xlabel('Number of Words')
plt.ylabel('Number of Reviews')
plt.legend()
plt.tight_layout()
plt.savefig('review_length_distribution.png', dpi=150)
plt.show()

# Print summary statistics
print("Mean review length:  ", round(df['review_length'].mean(), 1))
print("Median review length:", round(df['review_length'].median(), 1))
print("Max review length:   ", df['review_length'].max())
print("Min review length:   ", df['review_length'].min())
print("% reviews under 256 words:", round((df['review_length'] < 256).mean() * 100, 1), "%")

In [ ]:
!pip install wordcloud -q

#### 3. Word Cloud by Sentiment

In [ ]:
from wordcloud import WordCloud
import re

# Separate positive and negative reviews
positive_reviews = ' '.join(df[df['sentiment'] == 'positive']['review'])
negative_reviews = ' '.join(df[df['sentiment'] == 'negative']['review'])

# Remove HTML tags
positive_reviews = re.sub(r'<.*?>', '', positive_reviews)
negative_reviews = re.sub(r'<.*?>', '', negative_reviews)

# Generate word clouds
wc_positive = WordCloud(width=800, height=400, background_color='white',
                        colormap='Blues', max_words=100).generate(positive_reviews)

wc_negative = WordCloud(width=800, height=400, background_color='white',
                        colormap='Reds', max_words=100).generate(negative_reviews)

# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].imshow(wc_positive, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('Positive Reviews', fontsize=16, fontweight='bold')

axes[1].imshow(wc_negative, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('Negative Reviews', fontsize=16, fontweight='bold')

plt.suptitle('Most Frequent Words by Sentiment', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.savefig('wordcloud.png', dpi=150)
plt.show()

#### 4. Summary Statistics

In [ ]:
print("=" * 45)
print("        DATASET SUMMARY STATISTICS")
print("=" * 45)

print("\n--- General ---")
print(f"Total reviews:        {len(df):,}")
print(f"Total columns:        {df.shape[1]}")
print(f"Missing values:       {df.isnull().sum().sum()}")

print("\n--- Class Distribution ---")
print(f"Positive reviews:     {(df['sentiment'] == 'positive').sum():,}")
print(f"Negative reviews:     {(df['sentiment'] == 'negative').sum():,}")
print(f"Class balance ratio:  {(df['sentiment'] == 'positive').mean():.1%} / {(df['sentiment'] == 'negative').mean():.1%}")

print("\n--- Review Length (Word Count) ---")
print(f"Mean length:          {df['review_length'].mean():.1f} words")
print(f"Median length:        {df['review_length'].median():.1f} words")
print(f"Min length:           {df['review_length'].min()} words")
print(f"Max length:           {df['review_length'].max()} words")
print(f"Std deviation:        {df['review_length'].std():.1f} words")
print(f"Reviews <= 256 words: {(df['review_length'] <= 256).mean():.1%}")
print(f"Reviews >  256 words: {(df['review_length'] > 256).mean():.1%}")
print("=" * 45)